In [ ]:
# Finalised

import pandas as pd

# KERNEL SETTINGS
kernel = "RBF"

# Function to make candidates experimentally sensible, for the initial dataset
def make_experimental(raw_candidates):
    """Takes raw_candidates as a dataframe and returns experimentally plausible rounded values
    """
    reasonable_candidates = raw_candidates.copy()
    reasonable_candidates["Time"] = reasonable_candidates["Time"].round(decimals=0)
    reasonable_candidates["Na"] = reasonable_candidates["Na"].round(decimals=1)
    reasonable_candidates[["Cu", "Fe", "Mn", "Ni"]] = reasonable_candidates[["Cu", "Fe", "Mn", "Ni"]].round(decimals=4)
    return reasonable_candidates

In [ ]:
# RELEVANT DATASETS

qNIPV_run1 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qNIPV_bulk_candidates_run1.csv")
qNIPV_run2 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qNIPV_bulk_candidates_run2.csv")
qNIPV_run3 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qNIPV_bulk_candidates_run3.csv")
qNIPV = pd.concat([qNIPV_run1, qNIPV_run2, qNIPV_run3], axis=0)

qUCB_run1 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qUCB_bulk_candidates_run1.csv")
qUCB_run2 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qUCB_bulk_candidates_run2.csv")
qUCB_run3 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qUCB_bulk_candidates_run3.csv")
qUCB = pd.concat([qUCB_run1, qUCB_run2, qUCB_run3], axis=0)

qPSD_run1 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qPSD_bulk_candidates_run1.csv")
qPSD_run2 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qPSD_bulk_candidates_run2.csv")
qPSD_run3 = pd.read_csv(f"active_learning_model/active_learning_model_expr/{kernel}/{kernel}_whole_dataset_qPSD_bulk_candidates_run3.csv")
qPSD = pd.concat([qPSD_run1, qPSD_run2, qPSD_run3], axis=0)

df = pd.read_csv("active_learning_model/original_dataset/df_exp_semi_refined.csv")
df_success = df[df["Cat_Success"] == "Successful"]
cols = ["Time", "Na", "Cu", "Fe", "Mn", "Ni"]
initial_dataset = make_experimental(df_success[cols])

combined_df = pd.concat([qNIPV, qUCB, qPSD, initial_dataset], keys=['qNIPV', 'qUCB', 'qPSD', 'initial'], names=['Dataset']).reset_index()

combined_df.head()

,Dataset,level_1,Time,Na,Cu,Fe,Mn,Ni,pred_P2O3
0,qNIPV,0,5.0,0.6,0.6474,0.0145,0.2644,0.0737,0.95
1,qNIPV,1,6.0,0.8,0.1921,0.1449,0.4287,0.2344,1.04
2,qNIPV,2,8.0,0.9,0.0000,0.7495,0.0000,0.2505,0.05
3,qNIPV,3,8.0,0.9,0.0214,0.4102,0.0397,0.5286,0.53
4,qNIPV,4,8.0,0.8,0.2939,0.2289,0.0372,0.4400,0.72


In [11]:
# initial ternary plot without Time

import plotly.express as px

# No Cu
fig1 = px.scatter_ternary(combined_df, a='Fe', b='Mn', c='Ni',
              color='Dataset', title='No Cu')

# No Fe
fig2 = px.scatter_ternary(combined_df, a='Cu', b='Mn', c='Ni',
              color='Dataset', title='No Fe')

# No Mn
fig3 = px.scatter_ternary(combined_df, a='Cu', b='Fe', c='Ni',
              color='Dataset', title='No Mn')

# No Ni
fig4 = px.scatter_ternary(combined_df, a='Cu', b='Fe', c='Mn',
              color='Dataset', title='No Ni')

fig1.show()
fig2.show()
fig3.show()
fig4.show()

In [ ]:
# Prism plot of ternary diagram with Time as z axis value

import numpy as np
import plotly.graph_objects as go

# Set TMs
tms = ["Cu", "Fe", "Mn", "Ni"]

# Set z axis value
z_feature = "Time"
z_label = "Time (h)"

for i in range(len(tms)):

    other_tms = tms.copy()
    other_tms.remove(tms[i]) # removes a TM, length is 3

    # Make a ternary plot (equilateral triangle) in 2D cartesian
    total = combined_df[other_tms[0]] + combined_df[other_tms[1]] + combined_df[other_tms[2]]
    f_tm0 = combined_df[other_tms[0]] / total
    f_tm1 = combined_df[other_tms[1]] / total
    f_tm2 = combined_df[other_tms[2]] / total

    # Barycentric transformations such that tm0 vertex is (0, 0), tm1 vertex is (1, 0), tm2 vertex is (0.5, sqrt(3)/2)
    combined_df['X'] = f_tm1 + 0.5 * f_tm2
    combined_df['Y'] = f_tm2 * (np.sqrt(3) / 2)

    # Original dataset plotted in grey
    initial = combined_df[combined_df["Dataset"] == "initial"]
    qNIPV = combined_df[combined_df["Dataset"] == "qNIPV"]
    qUCB = combined_df[combined_df["Dataset"] == "qUCB"]
    qPSD = combined_df[combined_df["Dataset"] == "qPSD"]
    data = [go.Scatter3d(x=initial['X'], y=initial['Y'], z=initial[z_feature], mode='markers', name='Initial', marker=dict(size=2, color="grey", symbol="x"), opacity=0.8),
            go.Scatter3d(x=qNIPV['X'], y=qNIPV['Y'], z=qNIPV[z_feature], mode='markers', name='qNIPV', marker=dict(color='green', symbol='diamond'), opacity=0.8),
            go.Scatter3d(x=qUCB['X'], y=qUCB['Y'], z=qUCB[z_feature], mode='markers', name='qUCB', marker=dict(color='orange', symbol='square'), opacity=0.8),
            go.Scatter3d(x=qPSD['X'], y=qPSD['Y'], z=qPSD[z_feature], mode='markers', name='qPSD', marker=dict(color='purple', symbol='circle'), opacity=0.8)
    ]

    fig = go.Figure(data)

    # Prism outline using Cartesian coordinates of the 3 corners of the equilateral triangle
    corners = {
        other_tms[0]: (0.0, 0.0),
        other_tms[1]: (1.0, 0.0),
        other_tms[2]: (0.5, np.sqrt(3)/2)
    }
    z_min, z_max = combined_df[z_feature].min(), combined_df[z_feature].max()

    # Vertical wireframe line for each element corner from base to top
    for label, (cx, cy) in corners.items():
        fig.add_trace(go.Scatter3d(
            x=[cx, cx], y=[cy, cy], z=[z_min, z_max],
            mode='lines',
            line=dict(color='lightgray', width=3),
            name=f'{label} Axis Edge',
            showlegend=False
        ))

    # Lines betwen each corner at the base and top
    z_values = (z_min, z_max)
    for z in z_values:
        fig.add_trace(go.Scatter3d(
            x=[corners[other_tms[0]][0], corners[other_tms[1]][0], corners[other_tms[2]][0], corners[other_tms[0]][0]],
            y=[corners[other_tms[0]][1], corners[other_tms[1]][1], corners[other_tms[2]][1], corners[other_tms[0]][1]],
            z=[z, z, z, z],
            mode='lines',
            line=dict(color='lightgray', width=3),
            name='Base',
            showlegend=False
        ))

    # Position of corner labels
    centroid = np.array([0.5, np.sqrt(3)/6])
    scale = 1.25  # 25% larger triangle

    tm0_label = centroid + scale * (corners[other_tms[0]] - centroid)
    tm1_label = centroid + scale * (corners[other_tms[1]] - centroid)
    tm2_label = centroid + scale * (corners[other_tms[2]] - centroid)

    # Hide default 3D grid and add labels
    fig.update_layout(
        font=dict(family="Arial", size=14),
        scene=dict(

            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
            zaxis=dict(title=z_label, showgrid=False, showbackground=False),

            annotations=[
                dict(x=tm0_label[0], y=tm0_label[1], z=z_max, text=f"1.0 {other_tms[0]}", showarrow=False, font=dict(color="black")),
                dict(x=tm1_label[0], y=tm1_label[1], z=z_max, text=f"1.0 {other_tms[1]}", showarrow=False, font=dict(color="black")),
                dict(x=tm2_label[0], y=tm2_label[1], z=z_max, text=f"1.0 {other_tms[2]}", showarrow=False, font=dict(color="black")),
            ]
        ),
        title = f"No {tms[i]}",
        margin=dict(l=0, r=0, b=0, t=40),
    )

    fig.show()


In [ ]:
# Prism plot of ternary diagram with Na as z axis value

import numpy as np
import plotly.graph_objects as go

# Set TMs
tms = ["Cu", "Fe", "Mn", "Ni"]

# Set z axis value
z_feature = "Na"
z_label = "%Na"

for i in range(len(tms)):

    other_tms = tms.copy()
    other_tms.remove(tms[i]) # removes a TM, length is 3

    # Ternary plot (equilateral triangle) in 2D cartesian
    total = combined_df[other_tms[0]] + combined_df[other_tms[1]] + combined_df[other_tms[2]]
    f_tm0 = combined_df[other_tms[0]] / total
    f_tm1 = combined_df[other_tms[1]] / total
    f_tm2 = combined_df[other_tms[2]] / total

    # Barycentric transformations such that tm0 vertex is (0, 0), tm1 vertex is (1, 0), tm2 vertex is (0.5, sqrt(3)/2)
    combined_df['X'] = f_tm1 + 0.5 * f_tm2
    combined_df['Y'] = f_tm2 * (np.sqrt(3) / 2)

    # Original dataset plotted in grey
    initial = combined_df[combined_df["Dataset"] == "initial"]
    qNIPV = combined_df[combined_df["Dataset"] == "qNIPV"]
    qUCB = combined_df[combined_df["Dataset"] == "qUCB"]
    qPSD = combined_df[combined_df["Dataset"] == "qPSD"]
    data = [go.Scatter3d(x=initial['X'], y=initial['Y'], z=initial[z_feature], mode='markers', name='Initial', marker=dict(size=2, color="grey", symbol="x"), opacity=0.8),
            go.Scatter3d(x=qNIPV['X'], y=qNIPV['Y'], z=qNIPV[z_feature], mode='markers', name='qNIPV', marker=dict(color='green', symbol='diamond'), opacity=0.8),
            go.Scatter3d(x=qUCB['X'], y=qUCB['Y'], z=qUCB[z_feature], mode='markers', name='qUCB', marker=dict(color='orange', symbol='square'), opacity=0.8),
            go.Scatter3d(x=qPSD['X'], y=qPSD['Y'], z=qPSD[z_feature], mode='markers', name='qPSD', marker=dict(color='purple', symbol='circle'), opacity=0.8)
    ]

    fig = go.Figure(data)

    # Prism outline using Cartesian coordinates of the 3 corners of the equilateral triangle
    corners = {
        other_tms[0]: (0.0, 0.0),
        other_tms[1]: (1.0, 0.0),
        other_tms[2]: (0.5, np.sqrt(3)/2)
    }
    z_min, z_max = combined_df[z_feature].min(), combined_df[z_feature].max()

    # Vertical wireframe line for each element corner from base to top
    for label, (cx, cy) in corners.items():
        fig.add_trace(go.Scatter3d(
            x=[cx, cx], y=[cy, cy], z=[z_min, z_max],
            mode='lines',
            line=dict(color='lightgray', width=3),
            name=f'{label} Axis Edge',
            showlegend=False
        ))

    # Lines betwen each corner at the base and top
    z_values = (z_min, z_max)
    for z in z_values:
        fig.add_trace(go.Scatter3d(
            x=[corners[other_tms[0]][0], corners[other_tms[1]][0], corners[other_tms[2]][0], corners[other_tms[0]][0]],
            y=[corners[other_tms[0]][1], corners[other_tms[1]][1], corners[other_tms[2]][1], corners[other_tms[0]][1]],
            z=[z, z, z, z],
            mode='lines',
            line=dict(color='lightgray', width=3),
            name='Base',
            showlegend=False
        ))

    # Position of corner labels
    centroid = np.array([0.5, np.sqrt(3)/6])
    scale = 1.25

    tm0_label = centroid + scale * (corners[other_tms[0]] - centroid)
    tm1_label = centroid + scale * (corners[other_tms[1]] - centroid)
    tm2_label = centroid + scale * (corners[other_tms[2]] - centroid)

    # Hide default 3D grid and add labels
    fig.update_layout(
        font=dict(family="Arial", size=14),
        scene=dict(

            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
            zaxis=dict(title=z_label, showgrid=False, showbackground=False),

            annotations=[
                dict(x=tm0_label[0], y=tm0_label[1], z=z_max, text=f"1.0 {other_tms[0]}", showarrow=False, font=dict(color="black")),
                dict(x=tm1_label[0], y=tm1_label[1], z=z_max, text=f"1.0 {other_tms[1]}", showarrow=False, font=dict(color="black")),
                dict(x=tm2_label[0], y=tm2_label[1], z=z_max, text=f"1.0 {other_tms[2]}", showarrow=False, font=dict(color="black")),
            ]
        ),
        title = f"No {tms[i]}",
        margin=dict(l=0, r=0, b=0, t=40),
    )

    fig.show()


In [ ]:
# Get candidates from the batch of 50, for qNIPV only

# basic acquisition function settings (num_restarts=10, raw_samples=128)
qNIPV_50 = pd.read_csv("active_learning_model/active_learning_model_exp/RBF_qNIPV_bulk_candidates50.csv")

# enhanced acquisition function settings (num_restarts=20, raw_samples=256)
qNIPV_50_enhanced = pd.read_csv("active_learning_model/active_learning_model_exp/RBF_qNIPV_bulk_candidates50_enhanced.csv")

# new combined_df
combined_df_50 = pd.concat([qNIPV_50, qNIPV_50_enhanced, initial_dataset], keys=['qNIPV_basic', 'qNIPV_enhanced', 'Initial'], names=['Dataset']).reset_index()

In [ ]:
# initial ternary plot without Time, for qNIPV batch of 50

import plotly.express as px

# No Cu
fig1 = px.scatter_ternary(combined_df_50, a='Fe', b='Mn', c='Ni',
              color='Dataset', title='No Cu, qNIPV_50')

# No Fe
fig2 = px.scatter_ternary(combined_df_50, a='Cu', b='Mn', c='Ni',
              color='Dataset', title='No Fe, qNIPV_50')

# No Mn
fig3 = px.scatter_ternary(combined_df_50, a='Cu', b='Fe', c='Ni',
              color='Dataset', title='No Mn, qNIPV_50')

# No Ni
fig4 = px.scatter_ternary(combined_df_50, a='Cu', b='Fe', c='Mn',
              color='Dataset', title='No Ni, qNIPV_50')
fig1.show()
fig2.show()
fig3.show()
fig4.show()

In [ ]:
# Prism plot of ternary diagram with Time as z axis value, for batch of 50 instead

import numpy as np
import plotly.graph_objects as go

# make a ternary plot (equilateral triangle) using 2D cartesian coords
total = combined_df_50['Fe'] + combined_df_50['Mn'] + combined_df_50['Ni']
f_Fe = combined_df_50['Fe'] / total
f_Mn = combined_df_50['Mn'] / total
f_Ni = combined_df_50['Ni'] / total

combined_df_50['X'] = f_Mn + 0.5 * f_Ni
combined_df_50['Y'] = f_Ni * (np.sqrt(3) / 2)

# plot initial dataset in grey first
initial = combined_df_50[combined_df_50["Dataset"] == "Initial"]
qNIPV_50_basic_plot = combined_df_50[combined_df_50["Dataset"] == "qNIPV_basic"]
qNIPV_50_enhanced_plot = combined_df_50[combined_df_50["Dataset"] == "qNIPV_enhanced"]
data = [go.Scatter3d(x=initial['X'], y=initial['Y'], z=initial['Time'], mode='markers', name='Initial', marker=dict(size=2, color="grey", symbol="x"), opacity=0.8),
        go.Scatter3d(x=qNIPV_50_basic_plot['X'], y=qNIPV_50_basic_plot['Y'], z=qNIPV_50_basic_plot['Time'], mode='markers', name='qNIPV_basic', marker=dict(color='green', symbol='diamond'), opacity=0.8),
        go.Scatter3d(x=qNIPV_50_enhanced_plot['X'], y=qNIPV_50_enhanced_plot['Y'], z=qNIPV_50_enhanced_plot['Time'], mode='markers', name='qNIPV_enhanced', marker=dict(color='blue', symbol='circle'), opacity=0.8)
]

fig = go.Figure(data)

# prism outline
# cartesian coordinates of the 3 corners of the equilateral triangle
corners = {
    'Fe': (0.0, 0.0),
    'Mn': (1.0, 0.0),
    'Ni': (0.5, np.sqrt(3)/2)
}
z_min, z_max = combined_df['Time'].min(), combined_df['Time'].max()

# vertical wireframe line for each element corner from base to top
for label, (cx, cy) in corners.items():
    fig.add_trace(go.Scatter3d(
        x=[cx, cx], y=[cy, cy], z=[z_min, z_max],
        mode='lines',
        line=dict(color='lightgray', width=3),
        name=f'{label} Axis Edge',
        showlegend=False
    ))

# lines betwen each corner at the base and top
z_values = (z_min, z_max)
for z in z_values:
    fig.add_trace(go.Scatter3d(
        x=[corners['Fe'][0], corners['Mn'][0], corners['Ni'][0], corners['Fe'][0]],
        y=[corners['Fe'][1], corners['Mn'][1], corners['Ni'][1], corners['Fe'][1]],
        z=[z, z, z, z],
        mode='lines',
        line=dict(color='lightgray', width=3),
        name='Base',
        showlegend=False
    ))

# calculate position of corner labels
centroid = np.array([0.5, np.sqrt(3)/6])
scale = 1.25

Fe_label = centroid + scale * (corners["Fe"] - centroid)
Mn_label = centroid + scale * (corners["Mn"] - centroid)
Ni_label = centroid + scale * (corners["Ni"] - centroid)

# Update layout to get rid of default 3D grid
fig.update_layout(
    font=dict(family="Arial", size=14),
    scene=dict(
        # Hide the default grid box to highlight the triangular space
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='', showbackground=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='', showbackground=False),
        zaxis=dict(title='Time (h)', showgrid=False, showbackground=False),
        # Add labels at the physical positions of the triangle corners
        annotations=[
            dict(x=Fe_label[0], y=Fe_label[1], z=z_max, text=r"1.0 Fe", showarrow=False, font=dict(color="black")),
            dict(x=Mn_label[0], y=Mn_label[1], z=z_max, text="1.0 Mn", showarrow=False, font=dict(color="black")),
            dict(x=Ni_label[0], y=Ni_label[1], z=z_max, text="1.0 Ni", showarrow=False, font=dict(color="black")),
        ]
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()
